# EDA — Engineered Customer Churn Dataset
**50,000 rows · 14 features · Sri Lanka telecom/e-commerce market**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

df = pd.read_csv("../data/customers.csv")
print(f"Shape: {df.shape}")
print(f"Churn rate: {df.churned.mean():.1%}")
df.head()

## 1. Overall Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df["churned"].value_counts().rename({0:"Retained",1:"Churned"})
axes[0].bar(counts.index, counts.values, color=["#639922","#E24B4A"])
axes[0].set_title("Customer Count by Status")
churn_tier = df.groupby("product_tier")["churned"].mean() * 100
axes[1].bar(churn_tier.index, churn_tier.values, color=["#378ADD","#EF9F27","#639922"])
axes[1].set_title("Churn Rate % by Product Tier")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()
print(churn_tier.round(2))

## 2. KEY FINDING — Support Ticket Impact
> Customers with 3+ support tickets churn at **4x** the rate of 0-ticket customers.

In [ ]:
ticket_churn = df.groupby("support_tickets")["churned"].mean() * 100
plt.figure(figsize=(10, 5))
plt.bar(ticket_churn.index, ticket_churn.values,
        color=["#E24B4A" if x >= 3 else "#378ADD" for x in ticket_churn.index])
plt.axvline(x=2.5, color="black", linestyle="--", label="Intervention at ticket #2")
plt.xlabel("Number of Support Tickets")
plt.ylabel("Churn Rate %")
plt.title("Churn Rate by Support Ticket Count — RED = intervention zone")
plt.legend()
plt.show()
baseline = ticket_churn.iloc[0]
three_tick = ticket_churn.iloc[3]
print(f"0-ticket churn: {baseline:.1f}%  |  3-ticket churn: {three_tick:.1f}%  |  Multiplier: {three_tick/baseline:.1f}x")

## 3. Inactivity Cliff — Day 25 is the trigger point

In [ ]:
df["inactivity_bucket"] = pd.cut(df["last_login_days"],
    bins=[0,7,14,30,60,90,200], labels=["0-7d","8-14d","15-30d","31-60d","61-90d","90d+"])
inactivity_churn = df.groupby("inactivity_bucket")["churned"].mean() * 100
plt.figure(figsize=(10,5))
plt.plot(inactivity_churn.index.astype(str), inactivity_churn.values,
         marker="o", linewidth=2.5, color="#E24B4A", markersize=8)
plt.xlabel("Days Since Last Login")
plt.ylabel("Churn Rate %")
plt.title("Churn Rate by Inactivity Period — send re-engagement email at day 25")
plt.show()

## 4. Revenue at Risk Summary

In [ ]:
high_risk = df[(df.risk_score >= 60) & (df.churned == 0)]
print(f"High-risk ACTIVE customers: {len(high_risk):,}")
print(f"Monthly revenue at risk:    LKR {high_risk.monthly_spend.sum():,.0f}")
print(f"
Top 10 highest-risk customers:")
high_risk.sort_values("risk_score", ascending=False)[["customer_id","product_tier","monthly_spend","risk_score","support_tickets","last_login_days"]].head(10)